## Double Deep Q Network and Atari Games

In [ ]:
import gymnasium as gym
import numpy as np
import random
from collections import deque, namedtuple
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import image
from tqdm.notebook import trange
import seaborn as sns
import cv2  # OpenCV for image processing
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import ale_py

print("Before:", any(k.startswith("ALE/") for k in gym.registry.keys()))
gym.register_envs(ale_py)
print("After:", any(k.startswith("ALE/") for k in gym.registry.keys()))

In [ ]:
env = gym.make('ALE/Atlantis-v5', render_mode='rgb_array')

In [ ]:
env.reset()
frame = env.render()

In [ ]:
plt.imshow(frame)
plt.tight_layout()
plt.savefig('atlantis.png', dpi=300)

### Prioritised Experience Replay

In [ ]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

    def add(self, state, action, reward, next_state, done):
        max_prio = self.priorities.max() if self.buffer else 1.0
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.pos] = self.Transition(state, action, reward, next_state, done)
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
        probs = prios ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights /= weights.max()
        weights = np.array(weights, dtype=np.float32)

        batch = self.Transition(*zip(*samples))
        states = np.stack(batch.state)
        actions = np.array(batch.action)
        rewards = np.array(batch.reward)
        next_states = np.stack(batch.next_state)
        dones = np.array(batch.done)

        return states, actions, rewards, next_states, dones, indices, weights

    def update_priorities(self, batch_indices, batch_priorities):
        for idx, prio in zip(batch_indices, batch_priorities):
            self.priorities[idx] = prio

### Define the Q-Network

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, input_shape, n_actions):
        super(QNetwork, self).__init__()
        self.conv1 = nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        self.fc1 = nn.Linear(self.feature_size(input_shape), 512)
        self.fc2 = nn.Linear(512, n_actions)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

    def feature_size(self, input_shape):
        return self.conv3(self.conv2(self.conv1(torch.zeros(1, *input_shape)))).view(1, -1).size(1)

### Set & Parameters

In [ ]:
BATCH_SIZE = 256
GAMMA = 0.99
EPS_START = 1.0
EPS_END = 0.01
EPS_DECAY = 1000000
TARGET_UPDATE = 1000
MEMORY_CAPACITY = 10000
ALPHA = 0.6
BETA_START = 0.4
BETA_FRAMES = 100000
MAX_FRAMES = 100000
K = 4  # Number of successive frames to stack

In [ ]:
# Initialize environment and networks
n_actions = env.action_space.n
state_shape = (K, 84, 84)  # Use 84x84 as the square size
q_network = QNetwork(state_shape, n_actions).to(device)
target_network = QNetwork(state_shape, n_actions).to(device)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.Adam(q_network.parameters(), lr=0.0001)
replay_buffer = PrioritizedReplayBuffer(MEMORY_CAPACITY, alpha=ALPHA)

In [ ]:
def preprocess_frame(frame):
    # Convert to grayscale and resize the frame to 84x84
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (84, 84))
    frame = np.expand_dims(frame, axis=0)  # Add channel dimension
    return frame

In [ ]:
def select_action(state, eps):
    if random.random() > eps:
        with torch.no_grad():
            state = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = q_network(state)
            return q_values.max(1)[1].item()
    else:
        return random.randrange(n_actions)

### Training

In [ ]:
def compute_td_loss(batch_size, beta):
    states, actions, rewards, next_states, dones, indices, weights = replay_buffer.sample(batch_size, beta)

    states = torch.FloatTensor(states).to(device)
    next_states = torch.FloatTensor(next_states).to(device)
    actions = torch.LongTensor(actions).to(device)
    rewards = torch.FloatTensor(rewards).to(device)
    dones = torch.FloatTensor(dones).to(device)
    weights = torch.FloatTensor(weights).to(device)

    q_values = q_network(states)
    next_q_values = q_network(next_states)
    next_q_state_values = target_network(next_states)

    q_value = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
    next_q_value = next_q_state_values.gather(1, next_q_values.max(1)[1].unsqueeze(1)).squeeze(1)
    expected_q_value = rewards + GAMMA * next_q_value * (1 - dones)
    
    loss = (q_value - expected_q_value.detach()).pow(2) * weights
    prios = loss + 1e-5
    loss = loss.mean()

    optimizer.zero_grad()
    loss.backward()
    replay_buffer.update_priorities(indices, prios.data.cpu().numpy())
    optimizer.step()

    return loss

def update_target():
    target_network.load_state_dict(q_network.state_dict())

In [ ]:
eps = EPS_START
beta = BETA_START
frame_idx = 0
all_rewards = []
episode_reward = 0
state_deque = deque(maxlen=K)

# Initialize environment
state, _ = env.reset()
state_deque.append(preprocess_frame(state))

In [ ]:
while len(state_deque) < K:
    state_deque.append(state_deque[-1]) 

In [ ]:
for _ in trange(MAX_FRAMES):
    frame_idx += 1
    eps = max(EPS_END, EPS_START - frame_idx / EPS_DECAY)
    beta = min(1.0, BETA_START + frame_idx * (1.0 - BETA_START) / BETA_FRAMES)

    stacked_state = np.concatenate(state_deque, axis=0)
    action = select_action(stacked_state, eps)

    # Stochastic frame skipping
    skip = random.randint(2, 5)
    total_reward = 0
    for _ in range(skip):
        next_state, reward, done, _, _ = env.step(action)
        total_reward += reward
        if done:
            break

    next_state_processed = preprocess_frame(next_state)
    state_deque.append(next_state_processed)
    next_stacked_state = np.concatenate(state_deque, axis=0)

    replay_buffer.add(stacked_state, action, total_reward, next_stacked_state, done)

    state = next_state
    episode_reward += total_reward

    if done:
        state, _ = env.reset()
        state_deque = deque([preprocess_frame(state)] * K, maxlen=K)
        all_rewards.append(episode_reward)
        episode_reward = 0

    if len(replay_buffer.buffer) > BATCH_SIZE:
        loss = compute_td_loss(BATCH_SIZE, beta)

    if frame_idx % TARGET_UPDATE == 0:
        update_target()

    if frame_idx % 1000 == 0:
        print(f'Frame: {frame_idx}, Reward: {np.mean(all_rewards[-10:])}')

In [ ]:
torch.save(q_network.state_dict(), 'q_network_weights_atlantis.pth')

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(all_rewards,  'k-', label='Rewards')
plt.xlabel('Time Steps')
plt.ylabel('Reward')
plt.legend()
plt.grid(True)
plt.savefig('ddqn_rewards_atlantis.png', dpi=300)
plt.show()

In [ ]:
q_network.eval()
state, _ = env.reset()
state_deque = deque([preprocess_frame(state)] * K, maxlen=K)

In [ ]:
frames = []
done = False

while not done:
    stacked_state = np.concatenate(state_deque, axis=0)
    state_tensor = torch.FloatTensor(stacked_state).unsqueeze(0).to(device)
    with torch.no_grad():
        q_values = q_network(state_tensor)
    action = q_values.max(1)[1].item()
    
    next_state, _, done, _, _ = env.step(action)
    frames.append(env.render())
    
    next_state_processed = preprocess_frame(next_state)
    state_deque.append(next_state_processed)

env.close()

In [ ]:
# Function to update frame
def update_frame(num, frames, patch):
    patch.set_data(frames[num])
    return patch,

# Save frames as a GIF
fig = plt.figure()
patch = plt.imshow(frames[0])
plt.axis('off')

ani = FuncAnimation(fig, update_frame, frames=len(frames), fargs=(frames, patch), interval=50, blit=True)
ani.save("atlantis_episode.gif", writer=PillowWriter(fps=20))

plt.close(fig)

In [ ]:
display(Image(filename="atlantis_episode.gif"))

In [ ]:
def plot_frames(frames, n, m, a, b, filename):
    selected_frames = frames[::m][:n]
    fig, axes = plt.subplots(a, b, figsize=(20, 20))
    for i in range(a):
        for j in range(b):
            idx = i * b + j
            if idx < m:
                axes[i, j].imshow(selected_frames[idx])
                axes[i, j].axis('off')
            else:
                axes[i, j].axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
# Define the number of frames and the spacing
n = 9  # Number of frames to display
a = 3
b = 3
m = 100  # Frames apart

plot_frames(frames, n, m, a, b, 'atlantis_game.png')